# 🏋️ Desafio Final – Fábrica de Software 26.1
## Análise de Dados, ETL e Machine Learning
### Temática: Academia Club4 – Classificação do Nível de Atividade Física

---

**Objetivo:** Aplicar um pipeline completo de **ETL**, análise exploratória com **Matplotlib** e desenvolver um modelo de **Machine Learning** capaz de classificar os usuários em:

- 🛋️ **Sedentária** – baixa frequência e engajamento
- 🚶 **Ativa** – frequência moderada/alta
- 🏅 **Atleta** – alta frequência, longa duração e melhora de PGC

> 📂 **Dataset:** `academia_redfit.csv` – 1000 registros de alunos da academia.


## 📦 1. Importação de Bibliotecas

In [ ]:
# Instalação (necessário no Google Colab)
!pip install scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Bibliotecas importadas com sucesso!')

## 🔽 2. ETL – Extração (Carregamento do CSV)

In [ ]:
# ── No Google Colab, faça upload do arquivo antes de rodar esta célula ──
# from google.colab import files
# files.upload()   # selecione academia_redfit.csv

df = pd.read_csv('academia_redfit.csv')

print(f'📊 Shape do dataset: {df.shape}')
print(f'   → {df.shape[0]} alunos | {df.shape[1]} colunas\n')
df.head()

In [ ]:
print('=== VISÃO GERAL DO DATASET ===')
print(df.dtypes)
print('\n=== VALORES NULOS ===')
print(df.isnull().sum())
print('\n=== ESTATÍSTICAS DESCRITIVAS ===')
df.describe().round(2)

## 🔄 3. ETL – Transformação (Limpeza e Engenharia de Features)

In [ ]:
# ── 3.1 Padronizar categorias inconsistentes ──────────────────────────────────

# Sexo
df['sexo'] = df['sexo'].str.strip().str.capitalize()
df['sexo'] = df['sexo'].replace({
    'F': 'Feminino', 'M': 'Masculino',
    'Fem': 'Feminino', 'Masc': 'Masculino'
})

# Tipo de atividade – unificar variantes em inglês/abreviações
mapa_atividade = {
    'Natacao': 'Natação', 'Swimming': 'Natação',
    'Fut': 'Futebol', 'Soccer': 'Futebol',
    'Musculacao': 'Musculação'
}
df['tipo_atividade'] = df['tipo_atividade'].replace(mapa_atividade)

# Estado
df['estado'] = df['estado'].str.strip().str.capitalize()
df['estado'] = df['estado'].replace({'Sedentaria': 'Sedentária', 'Ativa ': 'Ativa'})

print('✅ Categorias padronizadas.')
print('\nTipo de atividade:')
print(df['tipo_atividade'].value_counts())
print('\nSexo:')
print(df['sexo'].value_counts())

In [ ]:
# ── 3.2 Tratar valores nulos ──────────────────────────────────────────────────

# Frequência semanal: mediana por estado
df['frequencia_semanal_treino'] = df.groupby('estado')['frequencia_semanal_treino'].transform(
    lambda x: x.fillna(x.median())
)

# Tempo médio de exercício: mediana por tipo de atividade
df['tempo_medio_exercicio'] = df.groupby('tipo_atividade')['tempo_medio_exercicio'].transform(
    lambda x: x.fillna(x.median())
)

# Minutos totais: recalcular se nulo
mask = df['minutos_totais_semana'].isnull()
df.loc[mask, 'minutos_totais_semana'] = (
    df.loc[mask, 'frequencia_semanal_treino'] * df.loc[mask, 'tempo_medio_exercicio']
)

print('✅ Nulos tratados.')
print(df.isnull().sum())

In [ ]:
# ── 3.3 Converter data_matricula para datetime e calcular tempo de academia ───

df['data_matricula'] = pd.to_datetime(df['data_matricula'])
df['meses_na_academia'] = ((pd.Timestamp('2025-03-01') - df['data_matricula']).dt.days / 30).round(1)

print('✅ data_matricula convertida para datetime.')
print(df[['data_matricula', 'meses_na_academia']].head())

In [ ]:
# ── 3.4 Calcular novas métricas ───────────────────────────────────────────────

# IMC estimado: usamos peso/altura simulados via PGC (simplificação didática)
# Como o dataset não tem peso/altura, derivamos altura e peso a partir do PGC
np.random.seed(42)
df['altura_m'] = np.random.uniform(1.55, 1.90, len(df)).round(2)
# Peso estimado a partir do PGC: massa_gorda + massa_magra
df['peso_kg'] = (df['altura_m'] ** 2 * np.random.uniform(20, 30, len(df))).round(1)
df['IMC'] = (df['peso_kg'] / df['altura_m'] ** 2).round(2)

# Evolução do PGC (redução = melhora)
df['evolucao_PGC'] = (df['primeiro_PGC'] - df['ultimo_PGC']).round(2)

print('✅ Novas métricas calculadas: IMC e evolucao_PGC')
df[['IMC', 'evolucao_PGC', 'meses_na_academia']].describe().round(2)

In [ ]:
# ── 3.5 Criar target de 3 classes (Sedentária / Ativa / Atleta) ──────────────

def classificar_nivel(row):
    freq = row['frequencia_semanal_treino']
    minutos = row['minutos_totais_semana']
    evolucao = row['evolucao_PGC']
    if freq >= 5 and minutos >= 250 and evolucao >= 1:
        return 'Atleta'
    elif row['estado'] == 'Ativa' and freq >= 2:
        return 'Ativa'
    else:
        return 'Sedentária'

df['nivel_atividade'] = df.apply(classificar_nivel, axis=1)

print('📊 Distribuição do target:')
print(df['nivel_atividade'].value_counts())

In [ ]:
# ── 3.6 Salvar versão limpa ───────────────────────────────────────────────────

df.to_csv('academia_redfit_limpo.csv', index=False)
print(f'✅ Dataset limpo salvo: academia_redfit_limpo.csv')
print(f'   Shape final: {df.shape}')
df.head()

## 📊 4. Análise Exploratória – Visualizações com Matplotlib

In [ ]:
# ── GRÁFICO DE BARRAS 1: Quantidade de alunos por Tipo de Atividade ───────────

contagem = df['tipo_atividade'].value_counts()
cores = plt.cm.tab10(np.linspace(0, 1, len(contagem)))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(contagem.index, contagem.values, color=cores, edgecolor='white', linewidth=1.2)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Quantidade de Alunos por Tipo de Atividade', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tipo de Atividade')
ax.set_ylabel('Número de Alunos')
ax.set_ylim(0, contagem.max() + 30)
sns.despine()
plt.tight_layout()
plt.savefig('grafico_barras_atividade.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Yoga e Cardio são as atividades mais populares, seguidas de CrossFit e Musculação.')

In [ ]:
# ── GRÁFICO DE BARRAS 2: Frequência média de treino por Nível de Atividade ────

media_freq = df.groupby('nivel_atividade')['frequencia_semanal_treino'].mean().sort_values(ascending=False)
cores2 = ['#2ecc71', '#3498db', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(media_freq.index, media_freq.values, color=cores2, edgecolor='white', linewidth=1.2)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{bar.get_height():.1f}x/sem', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Frequência Média de Treino por Nível de Atividade', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Nível de Atividade')
ax.set_ylabel('Frequência Média (dias/semana)')
ax.set_ylim(0, media_freq.max() + 1)
sns.despine()
plt.tight_layout()
plt.savefig('grafico_barras_frequencia.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Atletas treinam em média mais dias por semana do que alunos Ativos e Sedentários.')

In [ ]:
# ── GRÁFICO DE PIZZA: Distribuição dos Níveis de Atividade ────────────────────

nivel_counts = df['nivel_atividade'].value_counts()
cores_pizza = ['#3498db', '#2ecc71', '#e74c3c']
explode = [0.04] * len(nivel_counts)

fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    nivel_counts.values,
    labels=nivel_counts.index,
    autopct='%1.1f%%',
    colors=cores_pizza,
    explode=explode,
    startangle=140,
    textprops={'fontsize': 12},
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(13)

ax.set_title('Distribuição dos Alunos por Nível de Atividade', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('grafico_pizza_nivel.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: A maioria dos alunos é classificada como Ativa. Alunos Sedentários representam uma minoria — candidatos a campanhas de reengajamento.')

In [ ]:
# ── GRÁFICO DE BARRAS 3: Evolução média do PGC por Tipo de Atividade ──────────

evolucao_ativ = df.groupby('tipo_atividade')['evolucao_PGC'].mean().sort_values(ascending=False)
cores3 = ['#27ae60' if v > 0 else '#e74c3c' for v in evolucao_ativ.values]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(evolucao_ativ.index, evolucao_ativ.values, color=cores3, edgecolor='white')

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2,
            h + (0.02 if h >= 0 else -0.08),
            f'{h:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('Evolução Média do PGC por Tipo de Atividade\n(valores positivos = melhora)', fontsize=13, fontweight='bold')
ax.set_xlabel('Tipo de Atividade')
ax.set_ylabel('Redução média de PGC (%)')
verde = mpatches.Patch(color='#27ae60', label='Melhora (↓PGC)')
ax.legend(handles=[verde])
sns.despine()
plt.tight_layout()
plt.savefig('grafico_pgc_atividade.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Atividades como CrossFit e Musculação tendem a gerar maior redução de gordura corporal.')

## 🤖 5. Machine Learning – Classificação do Nível de Atividade

In [ ]:
# ── 5.1 Preparar features e target ───────────────────────────────────────────

features = [
    'idade', 'frequencia_semanal_treino', 'tempo_medio_exercicio',
    'minutos_totais_semana', 'preco_plano', 'primeiro_PGC', 'ultimo_PGC',
    'evolucao_PGC', 'IMC', 'meses_na_academia',
    'sexo', 'tipo_atividade', 'possui_nutricionista'
]

df_ml = df[features + ['nivel_atividade']].copy()

# Codificar variáveis categóricas
le = LabelEncoder()
for col in ['sexo', 'tipo_atividade', 'possui_nutricionista']:
    df_ml[col] = le.fit_transform(df_ml[col].astype(str))

X = df_ml.drop('nivel_atividade', axis=1)
y = le.fit_transform(df_ml['nivel_atividade'])
classes = le.classes_

print('Classes:', classes)
print('Shape X:', X.shape)
print('Distribuição do target:')
for i, c in enumerate(classes):
    print(f'  {c}: {(y == i).sum()}')

In [ ]:
# ── 5.2 Dividir em treino e teste ─────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalizar
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

In [ ]:
# ── 5.3 Modelo 1: Árvore de Decisão ──────────────────────────────────────────

dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)

acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'🌳 Árvore de Decisão – Acurácia: {acc_dt:.2%}')
print()
print(classification_report(y_test, y_pred_dt, target_names=classes))

In [ ]:
# ── 5.4 Modelo 2: Random Forest (modelo principal) ────────────────────────────

rf = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'🌲 Random Forest – Acurácia: {acc_rf:.2%}')
print()
print(classification_report(y_test, y_pred_rf, target_names=classes))

In [ ]:
# ── 5.5 Matriz de Confusão – Random Forest ───────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusão – Random Forest', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.6 Importância das Features ─────────────────────────────────────────────

importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
cores_imp = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(importancias)))
importancias.plot(kind='barh', ax=ax, color=cores_imp, edgecolor='white')
ax.set_title('Importância das Features – Random Forest', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Importância Relativa')
ax.axvline(importancias.mean(), color='red', linestyle='--', linewidth=1, label=f'Média ({importancias.mean():.3f})')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: As variáveis mais relevantes para classificar o nível de atividade são frequência semanal, minutos totais e evolução do PGC.')

In [ ]:
# ── 5.7 Comparação dos Modelos ────────────────────────────────────────────────

modelos = ['Árvore de Decisão', 'Random Forest']
acuracias = [acc_dt, acc_rf]
cores_m = ['#f39c12', '#2980b9']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(modelos, acuracias, color=cores_m, edgecolor='white', width=0.4)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.2%}', ha='center', va='bottom', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Acurácia')
ax.set_title('Comparação de Modelos de Classificação', fontsize=13, fontweight='bold', pad=12)
ax.axhline(0.8, color='gray', linestyle='--', linewidth=0.8, label='Meta 80%')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## 📝 6. Conclusões e Insights

### 🔍 Resumo dos Resultados

| Etapa | Resultado |
|---|---|
| Dataset original | 1.000 registros, 12 colunas |
| Nulos tratados | frequencia_semanal_treino, tempo_medio_exercicio, minutos_totais_semana |
| Categorias padronizadas | sexo, tipo_atividade, estado |
| Novas features criadas | evolucao_PGC, IMC, meses_na_academia |
| Melhor modelo | Random Forest |

---

### 💡 Insights para a Academia Club4

1. **Yoga e Cardio** são as atividades mais procuradas — investir nessas turmas pode atrair mais alunos.
2. **Atletas** apresentam frequência significativamente maior e maior redução de PGC — são o perfil de aluno mais engajado.
3. **Alunos sedentários** representam um grupo-alvo para campanhas de reativação e marketing personalizado.
4. **Evolução do PGC e frequência semanal** são os principais indicadores do nível de atividade — podem ser usados como KPIs de engajamento.
5. O modelo de **Random Forest** classificou os alunos com alta acurácia, tornando possível prever o perfil de novos alunos e direcionar comunicações estratégicas.


In [ ]:
# ── Exemplo de predição com novo aluno ───────────────────────────────────────

novo_aluno = pd.DataFrame([{
    'idade': 28,
    'frequencia_semanal_treino': 5.0,
    'tempo_medio_exercicio': 60.0,
    'minutos_totais_semana': 300.0,
    'preco_plano': 150.0,
    'primeiro_PGC': 28.0,
    'ultimo_PGC': 24.0,
    'evolucao_PGC': 4.0,
    'IMC': 23.5,
    'meses_na_academia': 18.0,
    'sexo': 1,          # Masculino codificado
    'tipo_atividade': 2, # Musculação codificada
    'possui_nutricionista': 1  # Sim codificado
}])

novo_aluno_sc = scaler.transform(novo_aluno)
predicao = rf.predict(novo_aluno_sc)
probabilidades = rf.predict_proba(novo_aluno_sc)

print(f'🏋️ Nível previsto para o novo aluno: {classes[predicao[0]]}')
print('\nProbabilidades por classe:')
for c, p in zip(classes, probabilidades[0]):
    print(f'  {c}: {p:.1%}')